# YOLOWorld — Learning Notebook

## What is YOLOWorld?

Standard YOLO models (like YOLOv8x-seg) are **closed-vocabulary** — they can only detect the 80 classes they were trained on. If you show them a lighter, they either ignore it or misclassify it as something else.

**YOLOWorld** is an **open-vocabulary** detection model. Instead of a fixed list of 80 classes, you describe what you want to detect in plain English — any word or phrase — and the model finds it. No retraining needed.

```
Standard YOLO:     "I know knife, person, bottle, car, ... (80 classes)"
YOLOWorld:         "Tell me what to look for → I'll find it"
```

---

## How does it work? (the key idea)

YOLOWorld uses **CLIP-style vision-language alignment**. During training, it learned to match visual features (shapes, textures, edges) with text embeddings (word meanings from a language model). The result: the model understands what words *mean* visually.

When you call `model.set_classes(["lighter", "pill bottle"])`, the model converts those words into text embeddings and uses them to search for matching visual patterns in the image — at inference time, without any weight updates.

```
Text:  "lighter"  ──text encoder──►  [0.12, -0.84, 0.33, ...]  (text embedding)
                                              ↕ similarity matching
Image: camera frame  ──CNN──►  [0.11, -0.81, 0.35, ...]  (visual embedding)
                                              ↕
                             High similarity → detected!
```

---

## YOLOWorld vs YOLOv8x-seg — when to use which

| Property | YOLOv8x-seg | YOLOWorld |
|---|---|---|
| Vocabulary | Fixed 80 COCO classes | Any text description |
| Training needed? | No (already trained) | No (zero-shot) |
| Detection accuracy | High (trained directly on class) | Lower (~50-70% for unusual items) |
| Segmentation masks | ✅ Yes — pixel-level | ❌ No — bounding boxes only |
| Speed | Slower (large seg head) | Faster |
| Best for | COCO objects (knife, person, bottle…) | Custom objects not in COCO |

**In our project:** We use both together. YOLOv8x-seg handles COCO classes with full mask accuracy; YOLOWorld handles the custom dangerous items (lighters, pill bottles, etc.) that COCO doesn't cover.

---

## What does `v2` mean in `yolov8x-worldv2.pt`?

YOLOWorld has two versions:

| Version | Key improvement |
|---|---|
| v1 | Original YOLOWorld |
| **v2** | Better text-image alignment, improved accuracy on ambiguous descriptions, supports more classes simultaneously |

Always use `v2` unless you have a specific reason not to.

---

## Model sizes

| Variant | Speed | Accuracy |
|---|---|---|
| `yolov8s-worldv2.pt` | Fastest | Lowest |
| `yolov8m-worldv2.pt` | Medium | Medium |
| `yolov8l-worldv2.pt` | Slow | Good |
| `yolov8x-worldv2.pt` | Slowest | **Best** ← we use this |

For a safety-critical application, always use `x`.

---
## Part 1 — Setup

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLOWorld

print("Imports OK")

---
## Part 2 — Load the Model and Set Classes

The critical difference from standard YOLO: **you must call `set_classes()` before inference**. This is what makes the model open-vocabulary — you're telling it what to search for.

### Rules for writing class descriptions

| What you write | Quality |
|---|---|
| `"lighter"` | Good — common word, unambiguous |
| `"disposable lighter"` | Better — more specific description |
| `"fire starting device"` | Worse — too abstract, model may not connect this to the visual pattern |
| `"pill bottle"` | Good |
| `"orange prescription pill bottle"` | Better — adds visual details the model can match |
| `"medicine"` | Worse — too abstract, many visual forms |

**Key principle:** Write descriptions that match how images are captioned on the internet — that's what the model learned from. Concrete, everyday nouns work best.

In [ ]:
model = YOLOWorld("yolov8x-worldv2.pt")   # downloads ~140 MB on first run

# Define what we want to detect
classes = [
    "lighter",
    "matches",
    "pill bottle",
    "medicine bottle",
    "button battery",
    "power cord",
    "extension cord",
    "plastic bag",
    "candle",
    "iron",
    "needle",
    "syringe",
    "cleaning spray bottle",
]

# This call reconfigures the model's output head in-place.
# After this, model.names maps 0..N-1 to your class descriptions.
model.set_classes(classes)

print("Classes configured:")
for id_, name in model.names.items():
    print(f"  {id_:>2}  {name}")

---
## Part 3 — Key Parameters

Most parameters are the same as YOLOv8 — but **`conf` needs special attention** for YOLOWorld.

### `conf` — must be lower than for trained models

A trained model has seen thousands of examples of each class. Its confidence scores for correct detections are typically 0.7–0.99.

YOLOWorld has never seen specific training examples of your custom classes. Even when it correctly identifies a lighter, the similarity score is often only 0.25–0.45. If you set `conf = 0.5`, you'll miss most real detections.

```
Trained model:   correct detection → conf ≈ 0.7–0.99  →  use conf=0.25 or higher
YOLOWorld:       correct detection → conf ≈ 0.20–0.45 →  use conf=0.15–0.25
```

**The trade-off:** Lower `conf` = more false positives. YOLOWorld with low conf may occasionally misidentify a remote control as a lighter. This is an inherent limitation of zero-shot detection — the reason Phase 2 (fine-tuning) exists.

### `iou` — same as standard YOLO

Default `0.45` works fine. YOLOWorld doesn't change NMS behaviour.

### `imgsz` — same as standard YOLO

Default `640`. Increase to `1280` for small distant objects.

### Number of classes in `set_classes()`

YOLOWorld can handle a large number of classes simultaneously, but:
- More classes → more computation (scales with N)
- More classes → slightly more false positives (more categories to potentially match)
- For best accuracy, include only classes you actually need
- Maximum practical limit: ~80–100 classes (beyond that, accuracy degrades)

---
## Part 4 — Run Inference on an Image

In [ ]:
# Load a test image
from ultralytics.utils import ASSETS
img_path = str(ASSETS / "bus.jpg")

# You can also test with your own photo of a dangerous item:
# img_path = "/path/to/photo_of_lighter.jpg"

img     = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title("Input image")
plt.axis("off")
plt.show()

In [ ]:
results = model.predict(
    source=img_path,
    conf=0.20,       # lower than standard YOLO — zero-shot scores are lower
    iou=0.45,
    imgsz=640,
    device="mps",
    verbose=True,
)

r = results[0]
print(f"\nDetections found: {len(r.boxes)}")

---
## Part 5 — Understanding YOLOWorld's Output

### What YOLOWorld returns vs what it doesn't

YOLOWorld **only returns bounding boxes** — no segmentation masks. This is a fundamental architectural difference: the open-vocabulary detection head is incompatible with the segmentation mask decoder.

```
YOLOv8x-seg:  boxes ✅   masks ✅   class scores ✅
YOLOWorld:    boxes ✅   masks ❌   class scores ✅
```

This is why contact detection for custom classes in our project falls back to **bounding box proximity** instead of pixel-level mask overlap.

In [ ]:
print("=== YOLOWorld Result Object ===")
print(f"r.boxes:  {r.boxes}   (always present)")
print(f"r.masks:  {r.masks}   ← always None for YOLOWorld")
print()

if len(r.boxes) == 0:
    print("No detections — the bus.jpg test image doesn't have our custom classes.")
    print("Try with a photo that contains a lighter, pill bottle, etc.")
else:
    for box in r.boxes:
        cls_id   = int(box.cls[0].item())
        cls_name = model.names[cls_id]         # your custom class name
        conf     = float(box.conf[0].item())
        xyxy     = [round(v) for v in box.xyxy[0].tolist()]
        print(f"  {cls_name:<25}  conf={conf:.3f}  box={xyxy}")

In [ ]:
# Visualise with .plot()
annotated = r.plot()

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("YOLOWorld detections (bounding boxes only, no masks)")
plt.axis("off")
plt.show()

---
## Part 6 — `set_classes()` In Depth

### You can call `set_classes()` multiple times — it's instant

Changing classes doesn't retrain the model. It simply recomputes the text embeddings for the new class list. This takes milliseconds.

In [ ]:
# Switch to detecting common objects for demonstration
model.set_classes(["person", "bus", "traffic light"])

res = model.predict(source=img_path, conf=0.20, device="mps", verbose=False)
print(f"With ['person', 'bus', 'traffic light']: {len(res[0].boxes)} detections")
for box in res[0].boxes:
    print(f"  {model.names[int(box.cls[0].item())]}  conf={float(box.conf[0].item()):.2f}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

# Switch back to dangerous items
model.set_classes(classes)
print("\nSwitched back to dangerous item classes.")

### Class description quality matters

Let's compare different ways to describe the same object.

In [ ]:
# Try different descriptions for the same concept — use your own test image for best results
description_variants = [
    ["lighter"],                      # short, common noun
    ["disposable lighter"],            # more specific
    ["cigarette lighter"],             # alternative common name
    ["fire starting device"],          # abstract — likely worse
]

print("Description comparison (bus.jpg — no lighters, so all should be 0 detections):")
print("Use a photo of a lighter for a meaningful test.\n")

for desc_list in description_variants:
    model.set_classes(desc_list)
    res = model.predict(source=img_path, conf=0.15, device="mps", verbose=False)
    n   = len(res[0].boxes)
    confs = [round(float(b.conf[0].item()), 3) for b in res[0].boxes]
    print(f"  {str(desc_list[0]):<30}  detections={n}  confs={confs}")

# Restore original classes
model.set_classes(classes)

---
## Part 7 — Confidence Threshold Deep Dive

### Why YOLOWorld's confidence scores are lower

This is the most important thing to understand when tuning YOLOWorld.

The confidence score is essentially a **cosine similarity** between the visual features the model extracted from a region of the image and the text embedding for your class description. This similarity is bounded by how well the two modalities align — and zero-shot alignment is always weaker than supervision from labelled training examples.

**Analogy:** Imagine describing a face to someone who has never seen that person vs. showing them a photo. The first method will always result in less certainty, even if the description is accurate.

In [ ]:
# Compare detection counts at different confidence thresholds
conf_values = [0.10, 0.15, 0.20, 0.30, 0.50]

print(f"{'conf':>6}  {'detections':>11}  {'class names'}")
print("-" * 60)

for conf_val in conf_values:
    res = model.predict(source=img_path, conf=conf_val, device="mps", verbose=False)
    boxes = res[0].boxes
    names = [model.names[int(b.cls[0].item())] for b in boxes]
    print(f"  {conf_val:>4}  {len(boxes):>11}  {', '.join(names) if names else '(none)'}")

print()
print("Note: bus.jpg has no dangerous items, so all non-zero detections are false positives.")
print("This illustrates why a too-low conf threshold causes false positives.")

---
## Part 8 — Proximity-Based Contact Detection

Since YOLOWorld provides no masks, our project uses **expanded bounding box proximity** to detect contact.

The idea: expand the dangerous item's bounding box outward by `PROXIMITY_MARGIN` pixels on all sides. If a person's bounding box overlaps this expanded zone, we consider contact to have occurred.

```
Original danger box:        Expanded box (margin=60px):
┌──────────┐                ┌───────────────────────────┐
│  lighter │                │         (60px buffer)     │
└──────────┘         →      │   ┌──────────┐            │
                            │   │  lighter │            │
                            │   └──────────┘            │
                            └───────────────────────────┘
```

A person entering the outer zone triggers the alert — even before physically touching the object. This is intentional: it gives parents a warning as the child *approaches* the item, not just when they grab it.

In [ ]:
def expand_box(box: tuple, margin: int, frame_w: int, frame_h: int) -> tuple:
    """Expand a bounding box outward by margin pixels, clamped to frame bounds."""
    x1, y1, x2, y2 = box
    return (max(0, x1 - margin), max(0, y1 - margin),
            min(frame_w, x2 + margin), min(frame_h, y2 + margin))

def boxes_overlap(a: tuple, b: tuple) -> bool:
    """True if two (x1,y1,x2,y2) rectangles intersect."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return not (ax2 < bx1 or bx2 < ax1 or ay2 < by1 or by2 < ay1)


# Example: simulate a detected lighter and a nearby person
frame_w, frame_h = 1280, 720
PROXIMITY_MARGIN = 60

danger_box = (500, 300, 560, 360)   # small lighter box
person_box = (400, 200, 650, 700)   # person overlapping with expanded zone

expanded = expand_box(danger_box, PROXIMITY_MARGIN, frame_w, frame_h)
contact  = boxes_overlap(person_box, expanded)

print(f"Danger box (lighter):   {danger_box}")
print(f"Expanded box (+60px):   {expanded}")
print(f"Person box:             {person_box}")
print(f"Overlap detected:       {contact}  ← {'ALERT!' if contact else 'safe'}")

# Visualise
canvas = np.zeros((frame_h, frame_w, 3), dtype=np.uint8)
ex1, ey1, ex2, ey2 = expanded
dx1, dy1, dx2, dy2 = danger_box
px1, py1, px2, py2 = person_box

cv2.rectangle(canvas, (ex1, ey1), (ex2, ey2), (0, 100, 200), 2)    # expanded zone (blue)
cv2.rectangle(canvas, (dx1, dy1), (dx2, dy2), (0, 0, 255), 3)       # danger item (red)
cv2.rectangle(canvas, (px1, py1), (px2, py2), (0, 200, 0), 2)       # person (green)

cv2.putText(canvas, "danger item", (dx1, dy1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
cv2.putText(canvas, "expanded zone", (ex1, ey1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 100, 200), 2)
cv2.putText(canvas, "person", (px1, py1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
if contact:
    cv2.putText(canvas, "CONTACT DETECTED", (400, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("Proximity-based contact detection (no masks needed)")
plt.axis("off")
plt.show()

---
## Part 9 — The Frame-Skipping Pattern

In our project, YOLOWorld runs every 3rd frame instead of every frame. This is a common optimisation for slower models in real-time applications.

In [ ]:
# Measure YOLOWorld inference time vs YOLOv8x-seg
from ultralytics import YOLO

seg_model = YOLO("yolov8x-seg.pt")

N = 5   # run each model N times and average

# Warm-up run (first run includes model initialisation overhead)
model.predict(source=img_path, device="mps", verbose=False)
seg_model.predict(source=img_path, device="mps", verbose=False)

# Time YOLOWorld
t0 = time.time()
for _ in range(N):
    model.predict(source=img_path, conf=0.20, device="mps", verbose=False)
world_ms = (time.time() - t0) / N * 1000

# Time YOLOv8x-seg
t0 = time.time()
for _ in range(N):
    seg_model.predict(source=img_path, conf=0.25, device="mps", verbose=False)
seg_ms = (time.time() - t0) / N * 1000

print(f"YOLOv8x-seg  avg: {seg_ms:.0f} ms/frame  (~{1000/seg_ms:.0f} FPS theoretical)")
print(f"YOLOWorld    avg: {world_ms:.0f} ms/frame  (~{1000/world_ms:.0f} FPS theoretical)")
print()
print("Frame-skipping strategy (WORLD_SKIP_FRAMES = 3):")
combined_ms = seg_ms + world_ms / 3
print(f"  Effective cost: {seg_ms:.0f} + {world_ms:.0f}/3 ≈ {combined_ms:.0f} ms/frame")
print(f"  Effective FPS: ~{1000/combined_ms:.0f}")
print()
print("This is acceptable because dangerous objects move slowly — a 3-frame-old")
print("detection of a lighter sitting on a table is still accurate.")

---
## Part 10 — Limitations of YOLOWorld (honest assessment)

Understanding these limitations is critical for setting expectations and motivating Phase 2.

In [ ]:
# Illustrate class confusion — classes that look similar may be confused

# These pairs are the most problematic in our project:
confusion_pairs = [
    ("pill bottle",      "bottle",       "Same shape, different label"),
    ("medicine bottle",  "bottle",       "Same shape, different label"),
    ("button battery",   "coin",         "Same circular shape, small size"),
    ("cleaning spray bottle", "spray bottle", "Nearly identical visual appearance"),
    ("needle",           "pin",          "Very thin elongated object"),
]

print("Known problematic class pairs for YOLOWorld zero-shot detection:\n")
print(f"  {'Target class':<25}  {'Often confused with':<25}  Reason")
print("  " + "-" * 80)
for target, confused, reason in confusion_pairs:
    print(f"  {target:<25}  {confused:<25}  {reason}")

print()
print("Root cause: these objects share visual features (shape, texture, size).")
print("YOLOWorld's text embeddings help, but cannot fully disambiguate visually")
print("similar objects without training data showing the subtle differences.")
print()
print("Solution: Phase 2 — collect images + fine-tune yolo11x-seg on these specific classes.")
print("A trained model learns exactly which features distinguish a pill bottle from")
print("a water bottle (label, cap type, colour, opacity, typical context).")

---
## Part 11 — Full Mini-Pipeline on a Single Image

Putting it all together: the same logic that runs in `yoloworld_demo.py`, but on a static image instead of a live camera feed.

In [ ]:
PROXIMITY_MARGIN = 60
PERSON_CLASS_ID  = 0   # in YOLOv8x-seg, person is class 0

# Step 1: Run YOLOv8x-seg to get persons (with masks)
seg_results = seg_model.predict(
    source=img_path, conf=0.25, device="mps", verbose=False
)
r_seg = seg_results[0]

# Step 2: Run YOLOWorld to detect custom dangerous items
model.set_classes(classes)
world_results = model.predict(
    source=img_path, conf=0.20, device="mps", verbose=False
)
r_world = world_results[0]

# Step 3: Extract person bounding boxes from seg results
person_boxes = []
if r_seg.boxes is not None:
    for box in r_seg.boxes:
        if seg_model.names[int(box.cls[0].item())] == "person":
            person_boxes.append(tuple(map(int, box.xyxy[0].tolist())))

# Step 4: Extract custom danger items from YOLOWorld results
danger_items = []
if r_world.boxes is not None:
    for box in r_world.boxes:
        cls_name = model.names[int(box.cls[0].item())]
        xyxy     = tuple(map(int, box.xyxy[0].tolist()))
        danger_items.append((cls_name, xyxy))

# Step 5: Check proximity for each danger item against each person
img_h, img_w = r_seg.orig_shape
contact_alerts = []

for (d_name, d_box) in danger_items:
    expanded = expand_box(d_box, PROXIMITY_MARGIN, img_w, img_h)
    for p_box in person_boxes:
        if boxes_overlap(p_box, expanded):
            contact_alerts.append(d_name)
            break

# Step 6: Draw and display results
annotated = r_seg.plot()   # draw seg model results as base

for (d_name, d_box) in danger_items:
    is_alert  = d_name in contact_alerts
    expanded  = expand_box(d_box, PROXIMITY_MARGIN, img_w, img_h)
    colour    = (0, 0, 255) if is_alert else (0, 165, 255)
    x1, y1, x2, y2 = d_box
    ex1, ey1, ex2, ey2 = expanded
    cv2.rectangle(annotated, (x1, y1), (x2, y2), colour, 2)
    cv2.rectangle(annotated, (ex1, ey1), (ex2, ey2), (0, 165, 255), 1)   # expanded zone
    label = f"{'DANGER: ' if is_alert else ''}{d_name}"
    cv2.putText(annotated, label, (x1, max(y1-6, 14)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, colour, 2)

print(f"Persons detected:        {len(person_boxes)}")
print(f"Danger items detected:   {len(danger_items)}")
print(f"Contact alerts:          {contact_alerts if contact_alerts else 'none'}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("Dual-model pipeline: YOLOv8x-seg (masks) + YOLOWorld (custom classes)")
plt.axis("off")
plt.show()

---
## Summary — YOLOWorld Key Takeaways

| Concept | Rule of thumb |
|---|---|
| When to use | Any object not in COCO 80 classes; rapid prototyping without training |
| `set_classes()` | Call once before inference; can switch class lists instantly |
| Description quality | Concrete nouns work best; avoid abstract phrases |
| `conf` threshold | Use 0.15–0.20 (lower than trained models — zero-shot scores are lower) |
| No masks | Contact detection falls back to bbox proximity — less accurate than mask overlap |
| Visually similar classes | YOLOWorld struggles with these; fine-tuning is the only real fix |
| Frame skipping | Run every 3rd frame to reduce GPU load; cache results between runs |
| Expected accuracy | ~50–70% detection rate for custom classes — good enough for Phase 1 validation |

### The fundamental trade-off

```
YOLOWorld:            No training needed   → lower accuracy   → good for Phase 1
Fine-tuned YOLO11x:   Training required    → higher accuracy  → Phase 3 goal
```

YOLOWorld lets us validate the entire detection + alert pipeline quickly and find which classes need training. The benchmark (Task 1.8) measures exactly this — which classes fall below 70% and are the priority targets for Phase 2 data collection.